<center><span style="font-size:40px;"><b>WORK FLOW IDEA</b></span></center>


We want to compare HMC vs NUTS when sampling from a known posterior, in this case a gaussian. So we need:
1. A model with a known posterior
2. Use *PyMC* to sample from it with both HMC and NUTS
3. Compare performance

#### STEP 1 — Generate synthetic data

We simulate data from a known distribution:
$$
x_i \sim \mathcal{N}(\mu_{\text{true}}, \sigma_{\text{true}})
$$

Conceptually we pretend the true parameters are unknown and we try to recover them via Bayesian inference.

#### STEP 2 — Define the model

Likelihood:

$$
p(x | \mu) = \prod_i \mathcal{N}(x_i | \mu, \sigma)
$$

Prior:

$$
\mu \sim \mathcal{N}(0, 10)
$$

Posterior:

$$
p(\mu | x) \propto p(x | \mu) p(\mu)
$$

This posterior is analytically tractable so we then can compare MCMC results with true posterior and check correctness.

#### STEP 3 — Define model in PyMC

Example:
```python
with pm.Model() as model:
    mu = pm.Normal("mu", mu=0, sigma=10)
    
    x_obs = pm.Normal("x_obs", mu=mu, sigma=sigma_true, observed=data)
```

#### STEP 4 — Run BOTH samplers

```python
# NUTS
trace_nuts = pm.sample(step=pm.NUTS(), draws=2000, tune=1000)

# HMC
trace_hmc = pm.sample(step=pm.HamiltonianMC(), draws=2000, tune=1000)
```

HMC requires tuning of the two parameters:
* step size $\epsilon $
* number of steps $L$

and we achieve it through varying:
```python
pm.HamiltonianMC(step_scale=0.1, path_length=1.0)
```

#### STEP 5 — Compare results

We compare posterior correctness (with analytical solutions):
* mean
* std

We compare convergence diagnostics (through ArviZ):

```python
az.summary(trace_nuts)
az.summary(trace_hmc)

# autocorrelation (the lower the better)
az.plot_autocorr(trace_nuts)
az.plot_autocorr(trace_hmc)

# trace plots
az.plot_trace(trace_nuts)
az.plot_trace(trace_hmc)
```

We look at:
* ESS (effective sample size)
* R-hat
* efficiency

$$
\text{Efficiency} = \frac{\text{ESS}}{\text{computation cost}}
$$

We can approximate:
* ESS / second
* ESS / number of samples

#### STEP 6 — Repeat with harder problem

Case 1: Unknown variance
$$
x_i \sim \mathcal{N}(\mu, \sigma), \quad \sigma \text{ unknown}
$$

Now:
* dimension = 2
* harder posterior

Case 2: Multivariate Gaussian (like in the paper)
$$
\theta \sim \mathcal{N}(0, \Sigma)
$

### CONCLUSIONS

We want to show that: 
* HMC:
    * sensitive to parameters ( \epsilon, L )
    * can perform poorly if badly tuned
* NUTS:
    * automatically adapts trajectory length
    * more robust

like what the paper claims.

---

---

---

<center><span style="font-size:40px;"><b>PARAMETERS AND ITERATIONS</b></span></center>

In the NUTS paper, they want to study:
- how well dual averaging works (i.e. tuning $\epsilon$ automatically)
- what value of $\delta$ (target acceptance rate) is best
- how HMC compares to NUTS

For each target distribution (they have 4 total) they run both HMC and NUTS.\
Each run has:
- total iterations = 2000
- first 1000 = warmup (adaptation phase)
- last 1000 = used for evaluation

During warmup:

- $\epsilon$ is adapted using dual averaging
- after warmup → $\epsilon$ is fixed

They fixed the dual averaging parameters: $ \gamma = 0.05, \quad t_0 = 10, \quad \kappa = 0.75 $. These control how $\epsilon$ is updated.

#### For HMC

They vary both:

- $\lambda$ = trajectory length (roughly $\epsilon \cdot L$)
- $\delta$ = target acceptance probability

Details:
- 10 values of $\lambda$: logarithmically spaced, largest = 40 × smallest, each step ≈ ×1.5
- 8 values of $\delta$: evenly spaced between 0.25 and 0.95

#### For NUTS

They vary only $\delta$.

Details: 15 values of $\delta$: evenly spaced between 0.25 and 0.95

---

For every configuration they **repeat the experiment 10 times**, each with a different random seed. This reduces randomness in results.

---

So in total they ran:

For HMC with:
- 4 target distributions
- 10 values of $\lambda$
- 8 values of $\delta$
- 10 random seeds

Total: $ 4 \times 10 \times 8 \times 10 = 3200 $ for HMC.

For NUTS with:
- 4 target distributions
- 15 values of $\delta$
- 10 random seeds

Total $4 \times 15 \times 10 = 600$ for NUTS.

#### JITTER IN HMC

After warmup, they modify HMC slightly. Instead of fixed $\epsilon$, at each iteration they they take $\epsilon \sim \text{Uniform}([0.9 \bar{\epsilon}, 1.1 \bar{\epsilon}])$. So at each iteration $\epsilon$ randomly varies ±10%.

This avoids pathological periodic trajectories, improves exploration and makes HMC more robust.

---

---

---

<center><span style="font-size:40px;"><b>EFFECTIVE SAMPLE SIZE</b></span></center>

MCMC gives us correlated samples $ \theta_1, \theta_2, \dots, \theta_M \sim p(\theta)$. These are not independent, so 1000 MCMC samples ≠ 1000 independent samples.

ESS answers: "how many independent samples would give the same estimation quality?"

Formally we estimate something like:

$$
\mathbb{E}_{p}[f(\theta)]
$$
 using the sample mean:

$$
\hat{\mu} = \frac{1}{M} \sum_{i=1}^M f(\theta_i)
$$

Because samples are correlated, variance is larger than the i.i.d. case.

ESS is defined so that:

$$
\text{Var}(\hat{\mu}_{\text{correlated}}) = \text{Var}(\hat{\mu}_{\text{iid with ESS samples}})
$$

So:

$$
\text{ESS} = \frac{M}{\text{autocorrelation penalty}}
$$

---

#### TERMS

- $\theta$ is the parameter vector you are sampling. In the paper (250D Gaussian): $\theta \in \mathbb{R}^{250}$. So each draw is a vector: $\theta_i = (\theta_i^{(1)}, \theta_i^{(2)}, \dots, \theta_i^{(250)})$.
- $p(\theta)$ is the target distribution, for example a multivariate Gaussian $p(\theta) \propto \exp\left(-\frac{1}{2} \theta^T A \theta \right)$
- $f(\theta)$ is the function you want to estimate the expectation of:
  - For estimating the mean: $f(\theta) = \theta$
  - For estimating uncertainty (second central moment): $f(\theta) = (\theta - \mu_{\text{true}})^2$, where $\mu_{\text{true}}$ is pre-calculated from a long baseline run.

---

#### ESS is computed per dimension

ESS is univariate, so it's about a single variable/dimension. But $\theta$ is a vector. So we take each component $\theta^{(j)}, j = 1, \dots, 250$, across all chains, treat them as parallel time series, and **compute a single multi-chain ESS separately for each dimension**.

---

#### ESS is computed for both first and second central moment

For each dimension $j$, we compute:
- $\text{ESS}_j^{(\text{mean})}$ using the identity function $f(\theta) = \theta$
- $\text{ESS}_j^{(\text{variance})}$ using the second central moment function $f(\theta) = (\theta - \mu_{\text{true}})^2$

Then we take the minimum of the two to obtain a conservative metric for that dimension:

$$
\text{ESS}_j = \min(\text{ESS}_j^{(\text{mean})}, \text{ESS}_j^{(\text{variance})})
$$

To ensure the sampler has adequately explored every single dimension of the 250D space, we take the minimum across all dimensions $\text{ESS}_1, \text{ESS}_2, \dots, \text{ESS}_{250}$:

$$
\text{ESS}_{\text{final}} = \min_j \text{ESS}_j
$$

---

#### CONCLUSION ON ESS COMPUTATION

When running an experiment with multiple chains (e.g., 4 chains):
1. We do **not** calculate ESS per chain and average them. 
2. Instead, we feed all 4 chains simultaneously into a multi-chain ESS estimator. This cross-chain analysis automatically penalizes the ESS if the chains failed to mix or converge to the same target.
3. This joint calculation yields exactly 250 values for $\text{ESS}^{(\text{mean})}$ and 250 values for $\text{ESS}^{(\text{variance})}$.
4. For each dimension, we take the lower of its mean and variance ESS.
5. Finally, we take the absolute minimum across all 250 dimensions, leaving us with **1 single conservative ESS number per experiment**.

### UNDERSTANDING `az.ess` IN ARVIZ

ArviZ provides the `az.ess()` function to estimate the Effective Sample Size. Modern versions of ArviZ do not just use the traditional classic ESS formula by default; instead, they implement the state-of-the-art **rank-normalized split-$\hat{R}$ and ESS** diagnostics introduced by Vehtari et al. (2021).

The function syntax is:
```python
import arviz as az
az.ess(inference_data, method="bulk")

```

---

#### EXPLANATION OF ARVIZ ESS METHODS

The `method` parameter allows you to choose exactly what type of estimation quality you want to measure:

1. **`method="mean"`**
* This calculates the **classic/traditional ESS** for the expected value.
* It maps exactly to $f(\theta) = \theta$.


2. **`method="sd"`**
* This calculates the ESS optimized for estimating the standard deviation (scale/uncertainty) of the target distribution.


3. **`method="bulk"` (The ArviZ Default)**
* **What it is:** It transforms the samples into ranks, normalizes them, and then computes the ESS.
* **Why use it:** It measures the sampling efficiency for the bulk (center) of the distribution. Because it operates on ranks, it is incredibly robust, stable, and works even for distributions that lack a finite mean or variance (like a Cauchy distribution). It is a direct "upgrade" to the mean ESS.


4. **`method="tail"`**
* **What it is:** It computes the ESS for two symmetric extreme quantiles (by default, the 5% and 95% quantiles) and returns the **minimum** of the two.
* **Why use it:** It specifically diagnoses whether the sampler is exploring the outer limits and tails of the distribution effectively, or if it is getting stuck.


5. **`method="quantile"`**
* Computes the ESS for a specific target quantile. You must pass the `prob` parameter (e.g., `prob=0.5` for the median).



---

#### LINK TO MEAN/VARIANCE ESS (NUTS PAPER VS. MODERN ARVIZ)

There is a direct conceptual bridge between what Hoffman & Gelman did in the 2014 NUTS paper and what ArviZ does today:

* **The Paper's $\text{ESS}^{(\text{mean})}$ $\longleftrightarrow$ ArviZ's `method="mean"` (or `method="bulk"`)**
* The paper uses the identity function to track how well the center/mean is estimated. ArviZ's `method="mean"` is mathematically identical to this. However, in modern workflows, `method="bulk"` is preferred because rank-normalization provides a more stable diagnostic for the same core zone.


* **The Paper's $\text{ESS}^{(\text{variance})}$ $\longleftrightarrow$ ArviZ's `method="sd"` (or `method="tail"`)**
* The paper uses the second central moment $f(\theta) = (\theta - \mu)^2$ to evaluate if the sampler can accurately capture uncertainty/scale. In ArviZ, `method="sd"` evaluates scale directly. Furthermore, `method="tail"` acts as a modern guardrail: if a sampler cannot accurately estimate the 5% and 95% boundaries, it cannot accurately estimate variance.



---

#### Step 1: Run the 50,000-sample Baseline
To compute the second central moment exactly like the paper, you need a precise baseline mean.
1. Run NUTS for 50,000 iterations on your target distribution (e.g., the 250D Gaussian).
2. Compute the mean vector across these samples. Let's call this 250-dimensional vector `mu_true`.

#### Step 2: Compute Strict Paper ESS using ArviZ
When you run your actual experiment comparison (HMC vs. NUTS) with fewer samples (e.g., 4 chains of 1000 post-warmup samples), you can extract the raw numpy arrays from your ArviZ `InferenceData` object to manually build the paper's metrics.

Here is how to write that loop in Python:

```python
import numpy as np
import arviz as az

# Assuming `idata` is your ArviZ InferenceData object from an experiment
# Shape of raw_samples will be (chains, draws, 250)
raw_samples = idata.posterior["theta"].values 

# --- 1. Compute Mean ESS ---
# method="mean" handles multi-chain joint variance automatically
ess_mean_per_dim = az.ess(idata, method="mean")["theta"].values # Shape: (250,)

# --- 2. Compute Second Central Moment (Variance) ESS ---
# Transform the samples manually using mu_true from your baseline long run
transformed_samples = (raw_samples - mu_true) ** 2

# Pass the transformed numpy array directly to ArviZ
ess_var_per_dim = az.ess(transformed_samples, method="mean") # Shape: (250,)

# --- 3. Minimize according to the paper ---
# Take the minimum of mean and variance for each dimension
ess_per_dim = np.minimum(ess_mean_per_dim, ess_var_per_dim)

# Take the absolute minimum across all 250 dimensions
final_paper_ess = np.min(ess_per_dim)

```

#### Step 3: Add a "Modern Comparison" Section to your Report

After showing the strict replication numbers using the code above, compute the modern ArviZ defaults:

```python
modern_bulk = np.min(az.ess(idata, method="bulk")["theta"].values)
modern_tail = np.min(az.ess(idata, method="tail")["theta"].values)
final_modern_ess = min(modern_bulk, modern_tail)

```

In your project report, you can conclude with a discussion paragraph like this:

> *"While Hoffman & Gelman (2014) monitored mixing using the minimum of the first and second central moments via traditional ESS equations, modern Bayesian workflows rely on rank-normalized Split-ESS (Vehtari et al., 2021). Our strict replication mapping shows that the original second central moment ESS tracks closely with modern `tail` ESS, while the original mean ESS is structurally upgraded by modern `bulk` ESS. Both approaches yield consistent conclusions when comparing HMC and NUTS over the 250D Gaussian distribution..."*